<a href="https://colab.research.google.com/github/WebosElegantes/Programaci-n-Sistemas-Base-1/blob/main/Copia_de_PSSB_I_Colab_U3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programación de Sistemas de Base I - Expresiones Regulares (ER) y Autómatas Finitos
**Asignatura:** Programación de Sistemas de Base I (8.º Semestre)  
**Unidad III:** Análisis Léxico  
**Tiempo estimado:** 3 horas (Asíncrono / Guiado)  
**Repositorio Base / Entrega:** Integración con Moodle / GitHub

---


## 1. Objetivos de Aprendizaje
1. Implementar una clase orientada a objetos que construya la matriz de transición y ejecuta la simulación paso a paso de cadenas de entrada.
2.  Implementar una clase orientada a objetos con algoritmos explícitos para el cálculo de la cerradura epsilon, la función move y la simulación paso a paso del algoritmo.
3. Implementar una arquitectura orientada a objetos para construir autómatas finitos usando el Algoritmo de Thompson para obtener el AFN-ɛ a partir de la ER.
4. Implementar el Algoritmo de Construcción de Subconjuntos utilizando programación orientada a objetos para convertir un AFN en un AFD.

---


## 2. Sección Práctica 1: Matriz de Transición
En esta sección analizaremos el algoritmo simulador para reconocimiento de cadenas mediante un AFD y la matriz de transición.


### [CÓDIGO 1.1] AFDeterminista.py
```python
# ==============================================================================
# Tema: Emulador de Autómata Finito Determinista (AFD) y Matriz de Transición
# ==============================================================================

from typing import Set, Dict, Tuple, List, Optional

class AutomataFinitoDeterminista:
    """Clase orientada a objetos para modelar y simular un AFD (5-tupla)."""

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], int],
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions  # Corregido: atributo asignado correctamente
        self.s0 = initial_state
        self.F = accepting_states
        
        # Estado de Error implícito
        self.ERROR_STATE = -1

    def transition(self, state: int, symbol: str) -> int:
        """Consulta la función/matriz de transición delta(s, c)."""
        return self.delta.get((state, symbol), self.ERROR_STATE) # Corregido: llamado al atributo delta

    def evaluate_string(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFD carácter a carácter trazando los estados."""
        current_state = self.s0
        trace = [f"Estado Inicial: q{current_state}"]

        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto Sigma.")
                return False, trace

            next_state = self.transition(current_state, char)
            if next_state == self.ERROR_STATE:
                trace.append(f"  [Bloqueo] Sin transición desde q{current_state} con '{char}'.")
                return False, trace

            trace.append(f"  Paso {i+1}: Símbolo '{char}' | Transición q{current_state} -> q{next_state}")
            current_state = next_state

        is_accepted = current_state in self.F
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(f"Estado Final: q{current_state} | Resultado: {status}")
        return is_accepted, trace

    def print_transition_matrix(self):
        """Imprime la representación tabular/matriz de transición del AFD."""
        sorted_symbols = sorted(list(self.Sigma))
        sorted_states = sorted(list(self.S))

        print("\n=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===")
        header = f"{'Estado':<8} | " + " | ".join([f"{sym:^5}" for sym in sorted_symbols])
        print(header)
        print("-" * len(header))

        for state in sorted_states:
            prefix = "-> " if state == self.s0 else (" * " if state in self.F else "   ")
            row_str = f"{prefix}q{state:<4} | "
            row_cells = []
            for sym in sorted_symbols:
                dest = self.transition(state, sym)
                cell = f"q{dest}" if dest != self.ERROR_STATE else "--"
                row_cells.append(f"{cell:^5}")
            row_str += " | ".join(row_cells)
            print(row_str)

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Construcción de un AFD para reconocer identificadores simples: Letra (L) seguida de Letras (L) o Dígitos (D)
    # Expresión regular equivalente: L (L | D)*
    
    states_set = {0, 1}
    alphabet_set = {'L', 'D'}
    
    # Transiciones:
    # q0 --L--> q1
    # q1 --L--> q1
    # q1 --D--> q1
    transitions_map = {
        (0, 'L'): 1,
        (1, 'L'): 1,
        (1, 'D'): 1
    }
    
    initial = 0
    accepting = {1}

    afd_id = AutomataFinitoDeterminista(states_set, alphabet_set, transitions_map, initial, accepting)

    # Mostrar la matriz de transición
    afd_id.print_transition_matrix()

    # Batería de pruebas
    test_inputs = ["LLD", "L1D", "DLL", "L"]
    
    print("\n=== EVALUACIÓN DE CADENAS DE PRUEBA ===")
    for test in test_inputs:
        # Convertir caracteres para simplificar la prueba (letras -> 'L', dígitos -> 'D')
        mapped_input = "".join(['L' if c.isalpha() else ('D' if c.isdigit() else c) for c in test])
        print(f"\nEvaluating Input Original: '{test}' -> Abstract: '{mapped_input}'")
        accepted, execution_trace = afd_id.evaluate_string(mapped_input)
        for line in execution_trace:
            print(line)
```



---

In [ ]:
from typing import Set, Dict, Tuple, List, Optional

class AutomataFinitoDeterminista:
    """Clase orientada a objetos para modelar y simular un AFD (5-tupla)."""

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], int],
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions  # Corregido: atributo asignado correctamente
        self.s0 = initial_state
        self.F = accepting_states

        # Estado de Error implícito
        self.ERROR_STATE = -1

    def transition(self, state: int, symbol: str) -> int:
        """Consulta la función/matriz de transición delta(s, c)."""
        return self.delta.get((state, symbol), self.ERROR_STATE) # Corregido: llamado al atributo delta

    def evaluate_string(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFD carácter a carácter trazando los estados."""
        current_state = self.s0
        trace = [f"Estado Inicial: q{current_state}"]

        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto Sigma.")
                return False, trace

            next_state = self.transition(current_state, char)
            if next_state == self.ERROR_STATE:
                trace.append(f"  [Bloqueo] Sin transición desde q{current_state} con '{char}'.")
                return False, trace

            trace.append(f"  Paso {i+1}: Símbolo '{char}' | Transición q{current_state} -> q{next_state}")
            current_state = next_state

        is_accepted = current_state in self.F
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(f"Estado Final: q{current_state} | Resultado: {status}")
        return is_accepted, trace

    def print_transition_matrix(self):
        """Imprime la representación tabular/matriz de transición del AFD."""
        sorted_symbols = sorted(list(self.Sigma))
        sorted_states = sorted(list(self.S))

        print("\n=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===")
        header = f"{'Estado':<8} | " + " | ".join([f"{sym:^5}" for sym in sorted_symbols])
        print(header)
        print("-" * len(header))

        for state in sorted_states:
            prefix = "-> " if state == self.s0 else (" * " if state in self.F else "   ")
            row_str = f"{prefix}q{state:<4} | "
            row_cells = []
            for sym in sorted_symbols:
                dest = self.transition(state, sym)
                cell = f"q{dest}" if dest != self.ERROR_STATE else "--"
                row_cells.append(f"{cell:^5}")
            row_str += " | ".join(row_cells)
            print(row_str)

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Construcción de un AFD para reconocer identificadores simples: Letra (L) seguida de Letras (L) o Dígitos (D)
    # Expresión regular equivalente: L (L | D)*

    states_set = {0, 1}
    alphabet_set = {'L', 'D'}

    # Transiciones:
    # q0 --L--> q1
    # q1 --L--> q1
    # q1 --D--> q1
    transitions_map = {
        (0, 'L'): 1,
        (1, 'L'): 1,
        (1, 'D'): 1
    }

    initial = 0
    accepting = {1}

    afd_id = AutomataFinitoDeterminista(states_set, alphabet_set, transitions_map, initial, accepting)

    # Mostrar la matriz de transición
    afd_id.print_transition_matrix()

    # Batería de pruebas
    test_inputs = ["LLD", "L1D", "DLL", "L"]

    print("\n=== EVALUACIÓN DE CADENAS DE PRUEBA ===")
    for test in test_inputs:
        # Convertir caracteres para simplificar la prueba (letras -> 'L', dígitos -> 'D')
        mapped_input = "".join(['L' if c.isalpha() else ('D' if c.isdigit() else c) for c in test])
        print(f"\nEvaluating Input Original: '{test}' -> Abstract: '{mapped_input}'")
        accepted, execution_trace = afd_id.evaluate_string(mapped_input)
        for line in execution_trace:
            print(line)


=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===
Estado   |   D   |   L  
------------------------
-> q0    |  --   |  q1  
 * q1    |  q1   |  q1  

=== EVALUACIÓN DE CADENAS DE PRUEBA ===

Evaluating Input Original: 'LLD' -> Abstract: 'LLL'
Estado Inicial: q0
  Paso 1: Símbolo 'L' | Transición q0 -> q1
  Paso 2: Símbolo 'L' | Transición q1 -> q1
  Paso 3: Símbolo 'L' | Transición q1 -> q1
Estado Final: q1 | Resultado: CADENA ACEPTADA

Evaluating Input Original: 'L1D' -> Abstract: 'LDL'
Estado Inicial: q0
  Paso 1: Símbolo 'L' | Transición q0 -> q1
  Paso 2: Símbolo 'D' | Transición q1 -> q1
  Paso 3: Símbolo 'L' | Transición q1 -> q1
Estado Final: q1 | Resultado: CADENA ACEPTADA

Evaluating Input Original: 'DLL' -> Abstract: 'LLL'
Estado Inicial: q0
  Paso 1: Símbolo 'L' | Transición q0 -> q1
  Paso 2: Símbolo 'L' | Transición q1 -> q1
  Paso 3: Símbolo 'L' | Transición q1 -> q1
Estado Final: q1 | Resultado: CADENA ACEPTADA

Evaluating Input Original: 'L' -> Abstract: 'L'
Estado Inicial: q0

## 3. Sección Práctica 2: Cerradura-ɛ
En esta sección construiremos una clase orientada a objetos con algoritmos explícitos para el cálculo de la cerradura epsilon y la función move.

### [CÓDIGO 2.1] epsilonClosure.py
```python
# ==============================================================================
# Tema: Emulador de Autómata Finito No Determinista con Transiciones Epsilon (AFN-e)
# ==============================================================================

from typing import Set, Dict, Tuple, List, Any

class NFA_Epsilon:
    """Clase orientada a objetos para modelar y simular un AFN con transiciones epsilon."""
    EPSILON = 'ε'  # Cadena vacía constante

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], Set[int]],
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions
        self.s0 = initial_state
        self.F = accepting_states

    def epsilon_closure_state(self, state: int) -> Set[int]:
        """Calcula la epsilon-cerradura para un solo estado usando un algoritmo de pila (DFS)."""
        closure = {state}
        stack = [state]

        while stack:
            current = stack.pop()
            # Buscar transiciones epsilon desde el estado actual
            next_states = self.delta.get((current, self.EPSILON), set())
            for nxt in next_states:
                if nxt not in closure:
                    closure.add(nxt)
                    stack.append(nxt)
        return closure

    def epsilon_closure_set(self, states_set: Set[int]) -> Set[int]:
        """Calcula la epsilon-cerradura para un conjunto de estados: e-closure(T) = U e-closure(s)."""
        total_closure = set()
        for state in states_set:
            total_closure.update(self.epsilon_closure_state(state))
        return total_closure

    def move(self, states_set: Set[int], symbol: str) -> Set[int]:
        """Calcula el conjunto mover(T, a): estados alcanzables con el símbolo 'a'."""
        reachable = set()
        for state in states_set:
            nxt = self.delta.get((state, symbol), set())
            reachable.update(nxt)
        return reachable

    def simulate(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFN (Algoritmo 3.22 del Dragon Book) sobre una cadena."""
        trace = []
        # Paso 1: Estado inicial activo S = e-closure(s0)
        current_active = self.epsilon_closure_state(self.s0)
        trace.append(f"Estado Activo Inicial e-closure({self.s0}): {sorted(list(current_active))}")

        # Bucle principal sobre cada carácter de la entrada
        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto.")
                return False, trace

            # Paso 2a: T = mover(S, c)
            move_set = self.move(current_active, char)
            # Paso 2b: S' = e-closure(T)
            next_active = self.epsilon_closure_set(move_set)

            trace.append(f"  Paso {i+1} ('{char}'): mover -> {sorted(list(move_set))} | e-closure -> {sorted(list(next_active))}")
            current_active = next_active

            if not current_active:
                trace.append("  [Bloqueo] El conjunto de estados activos quedó vacío.")
                return False, trace

        # Criterio de Aceptación: S ∩ F != Ø
        is_accepted = bool(current_active.intersection(self.F))
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(f"Estados Activos Finales: {sorted(list(current_active))} | Intersección F: {sorted(list(current_active.intersection(self.F)))}")
        trace.append(f"Resultado: {status}")

        return is_accepted, trace

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Construcción del AFN-e para el lenguaje (a|b)*abb
    states_all = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
    alphabet_in = {'a', 'b'}
    
    # Transiciones del AFN-e (basado en la figura 3.34 de Aho et al.)
    transitions_nfa = {
        (0, NFA_Epsilon.EPSILON): {1, 7},
        (1, NFA_Epsilon.EPSILON): {2, 4},
        (2, 'a'): {3},
        (3, NFA_Epsilon.EPSILON): {6},
        (4, 'b'): {5},
        (5, NFA_Epsilon.EPSILON): {6},
        (6, NFA_Epsilon.EPSILON): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}
    }
    
    start_st = 0
    accept_st = {10}

    afn = NFA_Epsilon(states_all, alphabet_in, transitions_nfa, start_st, accept_st)

    print("=== EMULADOR DE AFN-e (Simulación Directa) ===")
    test_string = "aabb"
    print(f"\nEvaluando la cadena: '{test_string}'")
    
    accepted, trace_log = afn.simulate(test_string)
    for line in trace_log:
        print(line)
```

In [ ]:
from typing import Set, Dict, Tuple, List, Any

class NFA_Epsilon:
    """Clase orientada a objetos para modelar y simular un AFN con transiciones epsilon."""
    EPSILON = 'ε'  # Cadena vacía constante

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], Set[int]],
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions
        self.s0 = initial_state
        self.F = accepting_states

    def epsilon_closure_state(self, state: int) -> Set[int]:
        """Calcula la epsilon-cerradura para un solo estado usando un algoritmo de pila (DFS)."""
        closure = {state}
        stack = [state]

        while stack:
            current = stack.pop()
            # Buscar transiciones epsilon desde el estado actual
            next_states = self.delta.get((current, self.EPSILON), set())
            for nxt in next_states:
                if nxt not in closure:
                    closure.add(nxt)
                    stack.append(nxt)
        return closure

    def epsilon_closure_set(self, states_set: Set[int]) -> Set[int]:
        """Calcula la epsilon-cerradura para un conjunto de estados: e-closure(T) = U e-closure(s)."""
        total_closure = set()
        for state in states_set:
            total_closure.update(self.epsilon_closure_state(state))
        return total_closure

    def move(self, states_set: Set[int], symbol: str) -> Set[int]:
        """Calcula el conjunto mover(T, a): estados alcanzables con el símbolo 'a'."""
        reachable = set()
        for state in states_set:
            nxt = self.delta.get((state, symbol), set())
            reachable.update(nxt)
        return reachable

    def simulate(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFN (Algoritmo 3.22 del Dragon Book) sobre una cadena."""
        trace = []
        # Paso 1: Estado inicial activo S = e-closure(s0)
        current_active = self.epsilon_closure_state(self.s0)
        trace.append(f"Estado Activo Inicial e-closure({self.s0}): {sorted(list(current_active))}")

        # Bucle principal sobre cada carácter de la entrada
        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto.")
                return False, trace

            # Paso 2a: T = mover(S, c)
            move_set = self.move(current_active, char)
            # Paso 2b: S' = e-closure(T)
            next_active = self.epsilon_closure_set(move_set)

            trace.append(f"  Paso {i+1} ('{char}'): mover -> {sorted(list(move_set))} | e-closure -> {sorted(list(next_active))}")
            current_active = next_active

            if not current_active:
                trace.append("  [Bloqueo] El conjunto de estados activos quedó vacío.")
                return False, trace

        # Criterio de Aceptación: S ∩ F != Ø
        is_accepted = bool(current_active.intersection(self.F))
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(f"Estados Activos Finales: {sorted(list(current_active))} | Intersección F: {sorted(list(current_active.intersection(self.F)))}")
        trace.append(f"Resultado: {status}")

        return is_accepted, trace

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Construcción del AFN-e para el lenguaje (a|b)*abb
    states_all = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
    alphabet_in = {'a', 'b'}

    # Transiciones del AFN-e (basado en la figura 3.34 de Aho et al.)
    transitions_nfa = {
        (0, NFA_Epsilon.EPSILON): {1, 7},
        (1, NFA_Epsilon.EPSILON): {2, 4},
        (2, 'a'): {3},
        (3, NFA_Epsilon.EPSILON): {6},
        (4, 'b'): {5},
        (5, NFA_Epsilon.EPSILON): {6},
        (6, NFA_Epsilon.EPSILON): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}
    }

    start_st = 0
    accept_st = {10}

    afn = NFA_Epsilon(states_all, alphabet_in, transitions_nfa, start_st, accept_st)

    print("=== EMULADOR DE AFN-e (Simulación Directa) ===")
    test_string = "aabb"
    print(f"\nEvaluando la cadena: '{test_string}'")

    accepted, trace_log = afn.simulate(test_string)
    for line in trace_log:
        print(line)

=== EMULADOR DE AFN-e (Simulación Directa) ===

Evaluando la cadena: 'aabb'
Estado Activo Inicial e-closure(0): [0, 1, 2, 4, 7]
  Paso 1 ('a'): mover -> [3, 8] | e-closure -> [1, 2, 3, 4, 6, 7, 8]
  Paso 2 ('a'): mover -> [3, 8] | e-closure -> [1, 2, 3, 4, 6, 7, 8]
  Paso 3 ('b'): mover -> [5, 9] | e-closure -> [1, 2, 4, 5, 6, 7, 9]
  Paso 4 ('b'): mover -> [5, 10] | e-closure -> [1, 2, 4, 5, 6, 7, 10]
Estados Activos Finales: [1, 2, 4, 5, 6, 7, 10] | Intersección F: [10]
Resultado: CADENA ACEPTADA


## 4. Sección Práctica 3: ER a AFN-ɛ
En esta sección implementaremos una arquitectura OO para obtener el AFN-ɛ a partir de la ER.

### [CÓDIGO 3.1] ERAFNepsilon.py
```python
# ==============================================================================
# Tema: Algoritmo de Construcción de Thompson
# ==============================================================================

from typing import Set, Dict, Tuple, List

class State:
    """Representa un estado individual de un autómata Thompson."""
    _id_counter = 0

    def __init__(self):
        self.id = State._id_counter
        State._id_counter += 1
        # Diccionario de transiciones: símbolo -> lista de estados destino
        self.transitions: Dict[str, List['State']] = {}

    def add_transition(self, symbol: str, target_state: 'State'):
        if symbol not in self.transitions:
            self.transitions[symbol] = []
        self.transitions[symbol].append(target_state)

    def __repr__(self):
        return f"q{self.id}"

class ThompsonNFA:
    """Encapsula un sub-autómata Thompson con exactamente UN inicio y UN fin."""
    EPSILON = 'ε'

    def __init__(self, start: State, accept: State):
        self.start = start
        self.accept = accept

    @staticmethod
    def build_symbol(symbol: str) -> 'ThompsonNFA':
        """Regla Base: Construcción para un símbolo a in Sigma (o epsilon)."""
        start = State()
        accept = State()
        start.add_transition(symbol, accept)
        return ThompsonNFA(start, accept)

    @staticmethod
    def build_union(nfa1: 'ThompsonNFA', nfa2: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 1: Unión (nfa1 | nfa2)."""
        start = State()
        accept = State()
        
        # Transiciones epsilon desde nuevo inicio a inicios de nfa1 y nfa2
        start.add_transition(ThompsonNFA.EPSILON, nfa1.start)
        start.add_transition(ThompsonNFA.EPSILON, nfa2.start)
        
        # Transiciones epsilon desde finales de nfa1 y nfa2 a nuevo final
        nfa1.accept.add_transition(ThompsonNFA.EPSILON, accept)
        nfa2.accept.add_transition(ThompsonNFA.EPSILON, accept)
        
        return ThompsonNFA(start, accept)

    @staticmethod
    def build_concat(nfa1: 'ThompsonNFA', nfa2: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 2: Concatenación (nfa1 . nfa2)."""
        # Conectar final de nfa1 con inicio de nfa2 mediante epsilon
        nfa1.accept.add_transition(ThompsonNFA.EPSILON, nfa2.start)
        return ThompsonNFA(nfa1.start, nfa2.accept)

    @staticmethod
    def build_kleene(nfa: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 3: Clausura de Kleene (nfa*)."""
        start = State()
        accept = State()
        
        # 1. Transición epsilon de bypass (0 repeticiones)
        start.add_transition(ThompsonNFA.EPSILON, accept)
        # 2. Transición epsilon hacia inicio del sub-autómata
        start.add_transition(ThompsonNFA.EPSILON, nfa.start)
        # 3. Transición epsilon desde final del sub-autómata hacia nuevo final
        nfa.accept.add_transition(ThompsonNFA.EPSILON, accept)
        # 4. Transición epsilon de retroceso (loop de repetición)
        nfa.accept.add_transition(ThompsonNFA.EPSILON, nfa.start)
        
        return ThompsonNFA(start, accept)

    def print_nfa(self):
        """Imprime la lista de adyacencia de todas las transiciones del autómata."""
        visited = set()
        stack = [self.start]
        print(f"\n=== AFN Thompson (Inicio: {self.start}, Aceptación: {self.accept}) ===")
        
        while stack:
            state = stack.pop()
            if state.id in visited:
                continue
            visited.add(state.id)
            
            for symbol, targets in state.transitions.items():
                for target in targets:
                    print(f"  {state} --({symbol})--> {target}")
                    if target.id not in visited:
                        stack.append(target)

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Reset contador de estados
    State._id_counter = 0

    # Construir inductivamente el AFN para la expresión: (a | b)*
    nfa_a = ThompsonNFA.build_symbol('a')      # a
    nfa_b = ThompsonNFA.build_symbol('b')      # b
    nfa_ab = ThompsonNFA.build_union(nfa_a, nfa_b)  # (a | b)
    nfa_star = ThompsonNFA.build_kleene(nfa_ab)     # (a | b)*

    nfa_star.print_nfa()
```


In [ ]:
from typing import Set, Dict, Tuple, List

class State:
    """Representa un estado individual de un autómata Thompson."""
    _id_counter = 0

    def __init__(self):
        self.id = State._id_counter
        State._id_counter += 1
        # Diccionario de transiciones: símbolo -> lista de estados destino
        self.transitions: Dict[str, List['State']] = {}

    def add_transition(self, symbol: str, target_state: 'State'):
        if symbol not in self.transitions:
            self.transitions[symbol] = []
        self.transitions[symbol].append(target_state)

    def __repr__(self):
        return f"q{self.id}"

class ThompsonNFA:
    """Encapsula un sub-autómata Thompson con exactamente UN inicio y UN fin."""
    EPSILON = 'ε'

    def __init__(self, start: State, accept: State):
        self.start = start
        self.accept = accept

    @staticmethod
    def build_symbol(symbol: str) -> 'ThompsonNFA':
        """Regla Base: Construcción para un símbolo a in Sigma (o epsilon)."""
        start = State()
        accept = State()
        start.add_transition(symbol, accept)
        return ThompsonNFA(start, accept)

    @staticmethod
    def build_union(nfa1: 'ThompsonNFA', nfa2: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 1: Unión (nfa1 | nfa2)."""
        start = State()
        accept = State()

        # Transiciones epsilon desde nuevo inicio a inicios de nfa1 y nfa2
        start.add_transition(ThompsonNFA.EPSILON, nfa1.start)
        start.add_transition(ThompsonNFA.EPSILON, nfa2.start)

        # Transiciones epsilon desde finales de nfa1 y nfa2 a nuevo final
        nfa1.accept.add_transition(ThompsonNFA.EPSILON, accept)
        nfa2.accept.add_transition(ThompsonNFA.EPSILON, accept)

        return ThompsonNFA(start, accept)

    @staticmethod
    def build_concat(nfa1: 'ThompsonNFA', nfa2: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 2: Concatenación (nfa1 . nfa2)."""
        # Conectar final de nfa1 con inicio de nfa2 mediante epsilon
        nfa1.accept.add_transition(ThompsonNFA.EPSILON, nfa2.start)
        return ThompsonNFA(nfa1.start, nfa2.accept)

    @staticmethod
    def build_kleene(nfa: 'ThompsonNFA') -> 'ThompsonNFA':
        """Regla Inductiva 3: Clausura de Kleene (nfa*)."""
        start = State()
        accept = State()

        # 1. Transición epsilon de bypass (0 repeticiones)
        start.add_transition(ThompsonNFA.EPSILON, accept)
        # 2. Transición epsilon hacia inicio del sub-autómata
        start.add_transition(ThompsonNFA.EPSILON, nfa.start)
        # 3. Transición epsilon desde final del sub-autómata hacia nuevo final
        nfa.accept.add_transition(ThompsonNFA.EPSILON, accept)
        # 4. Transición epsilon de retroceso (loop de repetición)
        nfa.accept.add_transition(ThompsonNFA.EPSILON, nfa.start)

        return ThompsonNFA(start, accept)

    def print_nfa(self):
        """Imprime la lista de adyacencia de todas las transiciones del autómata."""
        visited = set()
        stack = [self.start]
        print(f"\n=== AFN Thompson (Inicio: {self.start}, Aceptación: {self.accept}) ===")

        while stack:
            state = stack.pop()
            if state.id in visited:
                continue
            visited.add(state.id)

            for symbol, targets in state.transitions.items():
                for target in targets:
                    print(f"  {state} --({symbol})--> {target}")
                    if target.id not in visited:
                        stack.append(target)

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Reset contador de estados
    State._id_counter = 0

    # Construir inductivamente el AFN para la expresión: (a | b)*
    nfa_a = ThompsonNFA.build_symbol('a')      # a
    nfa_b = ThompsonNFA.build_symbol('b')      # b
    nfa_ab = ThompsonNFA.build_union(nfa_a, nfa_b)  # (a | b)
    nfa_star = ThompsonNFA.build_kleene(nfa_ab)     # (a | b)*

    nfa_star.print_nfa()


=== AFN Thompson (Inicio: q6, Aceptación: q7) ===
  q6 --(ε)--> q7
  q6 --(ε)--> q4
  q4 --(ε)--> q0
  q4 --(ε)--> q2
  q2 --(b)--> q3
  q3 --(ε)--> q5
  q5 --(ε)--> q7
  q5 --(ε)--> q4
  q0 --(a)--> q1
  q1 --(ε)--> q5


## 5. Sección Práctica 4: ER a AFN-ɛ
En esta sección implementaremos el Algoritmo de Construcción de Subconjuntos para convertir un AFN en un AFD.

### [CÓDIGO 4.1] AFN-AFD.py
```python
# ==============================================================================
# Tema: Algoritmo de Construcción de Subconjuntos
# ==============================================================================

from typing import Set, Dict, Tuple, List, Optional

class SubsetConstructionConverter:
    """Implementación formal de la Construcción de Subconjuntos (Algoritmo 3.20 Aho et al.)."""
    EPSILON = 'ε'

    def __init__(self, nfa_states: Set[int], alphabet: Set[str],
                 nfa_transitions: Dict[Tuple[int, str], Set[int]],
                 nfa_start_state: int, nfa_accept_states: Set[int]):
        self.nfa_states = nfa_states
        self.alphabet = alphabet
        self.nfa_delta = nfa_transitions
        self.nfa_start = nfa_start_state
        self.nfa_accept = nfa_accept_states

    def e_closure_state(self, state: int) -> Set[int]:
        """Calcula e-closure para un estado individual mediante algoritmo de pila."""
        closure = {state}
        stack = [state]
        while stack:
            curr = stack.pop()
            next_states = self.nfa_delta.get((curr, self.EPSILON), set())
            for nxt in next_states:
                if nxt not in closure:
                    closure.add(nxt)
                    stack.append(nxt)
        return closure

    def e_closure_set(self, states_set: Set[int]) -> Set[int]:
        """Calcula e-closure(T) = U e-closure(s) para s in T."""
        total = set()
        for s in states_set:
            total.update(self.e_closure_state(s))
        return total

    def move(self, states_set: Set[int], symbol: str) -> Set[int]:
        """Calcula mover(T, a): estados alcanzables con 'a' desde T."""
        reachable = set()
        for s in states_set:
            nxt = self.nfa_delta.get((s, symbol), set())
            reachable.update(nxt)
        return reachable

    def convert(self) -> Tuple[List[Set[int]], Dict[Tuple[int, str], int], Set[int]]:
        """Ejecuta el Algoritmo de Construcción de Subconjuntos."""
        # 1. Estado Inicial del AFD: A = e-closure(s0)
        start_closure = self.e_closure_state(self.nfa_start)
        
        d_states: List[Set[int]] = [start_closure]  # Lista de conjuntos Dstates
        d_tran: Dict[Tuple[int, str], int] = {}     # Matriz Dtran[state_idx, symbol]
        unmarked_indices = [0]                      # Cola de estados no marcados

        while unmarked_indices:
            curr_idx = unmarked_indices.pop(0)
            curr_set = d_states[curr_idx]

            for symbol in sorted(list(self.alphabet)):
                # U = e-closure(mover(T, a))
                move_res = self.move(curr_set, symbol)
                u_set = self.e_closure_set(move_res)

                if not u_set:
                    continue  # Transición vacía / no existe

                # Comprobar si U ya existe en Destados
                if u_set in d_states:
                    target_idx = d_states.index(u_set)
                else:
                    d_states.append(u_set)
                    target_idx = len(d_states) - 1
                    unmarked_indices.append(target_idx)

                d_tran[(curr_idx, symbol)] = target_idx

        # 2. Identificar estados finales del AFD (los que contienen al menos un final del AFN)
        dfa_accept_states = set()
        for idx, d_set in enumerate(d_states):
            if bool(d_set.intersection(self.nfa_accept)):
                dfa_accept_states.add(idx)

        return d_states, d_tran, dfa_accept_states

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # AFN para (a|b)*abb (Ejemplo 3.21 de Aho et al.)
    nfa_S = set(range(11))
    nfa_Sigma = {'a', 'b'}
    nfa_transitions_map = {
        (0, 'ε'): {1, 7},
        (1, 'ε'): {2, 4},
        (2, 'a'): {3},
        (3, 'ε'): {6},
        (4, 'b'): {5},
        (5, 'ε'): {6},
        (6, 'ε'): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}
    }
    nfa_s0 = 0
    nfa_F = {10}

    converter = SubsetConstructionConverter(nfa_S, nfa_Sigma, nfa_transitions_map, nfa_s0, nfa_F)
    dfa_nodes, dfa_matrix, dfa_accept = converter.convert()

    print("=== RESULTADO DE LA CONSTRUCCIÓN DE SUBCONJUNTOS ===")
    print(f"Total de estados AFD generados: {len(dfa_nodes)}\n")

    state_labels = [chr(65 + i) for i in range(len(dfa_nodes))]  # Nombres A, B, C, D, E...

    for idx, node_set in enumerate(dfa_nodes):
        is_acc = " [ACEPTACIÓN]" if idx in dfa_accept else ""
        print(f"Estado AFD {state_labels[idx]} ({idx}): {sorted(list(node_set))}{is_acc}")

    print("\n=== MATRIZ DE TRANSICIÓN Dtran ===")
    print(f"{'Estado Orig':<12} | {'Símbolo':<8} | {'Estado Destino':<12}")
    print("-" * 40)
    for (src_idx, sym), tgt_idx in dfa_matrix.items():
        print(f"{state_labels[src_idx]} ({src_idx}){'':<6} | {sym:<8} | {state_labels[tgt_idx]} ({tgt_idx})")
```

In [ ]:
from typing import Set, Dict, Tuple, List, Optional

class SubsetConstructionConverter:
    """Implementación formal de la Construcción de Subconjuntos (Algoritmo 3.20 Aho et al.)."""
    EPSILON = 'ε'

    def __init__(self, nfa_states: Set[int], alphabet: Set[str],
                 nfa_transitions: Dict[Tuple[int, str], Set[int]],
                 nfa_start_state: int, nfa_accept_states: Set[int]):
        self.nfa_states = nfa_states
        self.alphabet = alphabet
        self.nfa_delta = nfa_transitions
        self.nfa_start = nfa_start_state
        self.nfa_accept = nfa_accept_states

    def e_closure_state(self, state: int) -> Set[int]:
        """Calcula e-closure para un estado individual mediante algoritmo de pila."""
        closure = {state}
        stack = [state]
        while stack:
            curr = stack.pop()
            next_states = self.nfa_delta.get((curr, self.EPSILON), set())
            for nxt in next_states:
                if nxt not in closure:
                    closure.add(nxt)
                    stack.append(nxt)
        return closure

    def e_closure_set(self, states_set: Set[int]) -> Set[int]:
        """Calcula e-closure(T) = U e-closure(s) para s in T."""
        total = set()
        for s in states_set:
            total.update(self.e_closure_state(s))
        return total

    def move(self, states_set: Set[int], symbol: str) -> Set[int]:
        """Calcula mover(T, a): estados alcanzables con 'a' desde T."""
        reachable = set()
        for s in states_set:
            nxt = self.nfa_delta.get((s, symbol), set())
            reachable.update(nxt)
        return reachable

    def convert(self) -> Tuple[List[Set[int]], Dict[Tuple[int, str], int], Set[int]]:
        """Ejecuta el Algoritmo de Construcción de Subconjuntos."""
        # 1. Estado Inicial del AFD: A = e-closure(s0)
        start_closure = self.e_closure_state(self.nfa_start)

        d_states: List[Set[int]] = [start_closure]  # Lista de conjuntos Dstates
        d_tran: Dict[Tuple[int, str], int] = {}     # Matriz Dtran[state_idx, symbol]
        unmarked_indices = [0]                      # Cola de estados no marcados

        while unmarked_indices:
            curr_idx = unmarked_indices.pop(0)
            curr_set = d_states[curr_idx]

            for symbol in sorted(list(self.alphabet)):
                # U = e-closure(mover(T, a))
                move_res = self.move(curr_set, symbol)
                u_set = self.e_closure_set(move_res)

                if not u_set:
                    continue  # Transición vacía / no existe

                # Comprobar si U ya existe en Destados
                if u_set in d_states:
                    target_idx = d_states.index(u_set)
                else:
                    d_states.append(u_set)
                    target_idx = len(d_states) - 1
                    unmarked_indices.append(target_idx)

                d_tran[(curr_idx, symbol)] = target_idx

        # 2. Identificar estados finales del AFD (los que contienen al menos un final del AFN)
        dfa_accept_states = set()
        for idx, d_set in enumerate(d_states):
            if bool(d_set.intersection(self.nfa_accept)):
                dfa_accept_states.add(idx)

        return d_states, d_tran, dfa_accept_states

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # AFN para (a|b)*abb (Ejemplo 3.21 de Aho et al.)
    nfa_S = set(range(11))
    nfa_Sigma = {'a', 'b'}
    nfa_transitions_map = {
        (0, 'ε'): {1, 7},
        (1, 'ε'): {2, 4},
        (2, 'a'): {3},
        (3, 'ε'): {6},
        (4, 'b'): {5},
        (5, 'ε'): {6},
        (6, 'ε'): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}
    }
    nfa_s0 = 0
    nfa_F = {10}

    converter = SubsetConstructionConverter(nfa_S, nfa_Sigma, nfa_transitions_map, nfa_s0, nfa_F)
    dfa_nodes, dfa_matrix, dfa_accept = converter.convert()

    print("=== RESULTADO DE LA CONSTRUCCIÓN DE SUBCONJUNTOS ===")
    print(f"Total de estados AFD generados: {len(dfa_nodes)}\n")

    state_labels = [chr(65 + i) for i in range(len(dfa_nodes))]  # Nombres A, B, C, D, E...

    for idx, node_set in enumerate(dfa_nodes):
        is_acc = " [ACEPTACIÓN]" if idx in dfa_accept else ""
        print(f"Estado AFD {state_labels[idx]} ({idx}): {sorted(list(node_set))}{is_acc}")

    print("\n=== MATRIZ DE TRANSICIÓN Dtran ===")
    print(f"{'Estado Orig':<12} | {'Símbolo':<8} | {'Estado Destino':<12}")
    print("-" * 40)
    for (src_idx, sym), tgt_idx in dfa_matrix.items():
        print(f"{state_labels[src_idx]} ({src_idx}){'':<6} | {sym:<8} | {state_labels[tgt_idx]} ({tgt_idx})")

=== RESULTADO DE LA CONSTRUCCIÓN DE SUBCONJUNTOS ===
Total de estados AFD generados: 5

Estado AFD A (0): [0, 1, 2, 4, 7]
Estado AFD B (1): [1, 2, 3, 4, 6, 7, 8]
Estado AFD C (2): [1, 2, 4, 5, 6, 7]
Estado AFD D (3): [1, 2, 4, 5, 6, 7, 9]
Estado AFD E (4): [1, 2, 4, 5, 6, 7, 10] [ACEPTACIÓN]

=== MATRIZ DE TRANSICIÓN Dtran ===
Estado Orig  | Símbolo  | Estado Destino
----------------------------------------
A (0)       | a        | B (1)
A (0)       | b        | C (2)
B (1)       | a        | B (1)
B (1)       | b        | D (3)
C (2)       | a        | B (1)
C (2)       | b        | C (2)
D (3)       | a        | B (1)
D (3)       | b        | E (4)
E (4)       | a        | B (1)
E (4)       | b        | C (2)


## Desafío 1: Exploración y Modificación de un AFD (Nivel: Básico)
Script de referencia: AFDeterminista.py

**Objetivo:** Comprender la estructura de una 5-tupla, el manejo de diccionarios como funciones de transición y la representación tabular.

**Consigna para el estudiante:**

>Ejecución y análisis: Capture y ejecute el programa original. Observe detenidamente cómo se imprime la matriz de transición (`Dtran`) y cómo se evalúan las cadenas `"LLD"`, `"L1D"`, etc.

>Modificación del alfabeto y el lenguaje: Modifique el autómata para que, además de reconocer identificadores simples (`L(L|D)*`), acepte también números enteros positivos (por ejemplo, una secuencia de uno o más dígitos: `D+`).

>Ampliación de pruebas: Agregue al arreglo `test_inputs` al menos tres cadenas adicionales (una válida que comience con dígitos, otra mixta y una inválida) y documente paso a paso mediante comentarios en su código por qué el AFD rechaza o acepta cada una.

In [ ]:
# ==============================================================================
# Tema: Desafío 1 - Emulador de Autómata Finito Determinista (AFD) modificado
# ==============================================================================

from typing import Set, Dict, Tuple, List, Optional

class AutomataFinitoDeterminista:
    """Clase orientada a objetos para modelar y simular un AFD (5-tupla)."""

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], int],
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions
        self.s0 = initial_state
        self.F = accepting_states

        # Estado de Error implícito
        self.ERROR_STATE = -1

    def transition(self, state: int, symbol: str) -> int:
        """Consulta la función/matriz de transición delta(s, c)."""
        return self.delta.get((state, symbol), self.ERROR_STATE)

    def evaluate_string(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFD carácter a carácter trazando los estados."""
        current_state = self.s0
        trace = [f"Estado Inicial: q{current_state}"]

        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                trace.append(f"   [Error] Carácter '{char}' fuera del alfabeto Sigma.")
                return False, trace

            next_state = self.transition(current_state, char)
            if next_state == self.ERROR_STATE:
                trace.append(f"   [Bloqueo] Sin transición desde q{current_state} con '{char}'.")
                return False, trace

            trace.append(f"   Paso {i+1}: Símbolo '{char}' | Transición q{current_state} -> q{next_state}")
            current_state = next_state

        is_accepted = current_state in self.F
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(f"Estado Final: q{current_state} | Resultado: {status}")
        return is_accepted, trace

    def print_transition_matrix(self):
        """Imprime la representación tabular/matriz de transición del AFD."""
        sorted_symbols = sorted(list(self.Sigma))
        sorted_states = sorted(list(self.S))

        print("\n=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===")
        header = f"{'Estado':<8} | " + " | ".join([f"{sym:^5}" for sym in sorted_symbols])
        print(header)
        print("-" * len(header))

        for state in sorted_states:
            prefix = "-> " if state == self.s0 else (" * " if state in self.F else "   ")
            row_str = f"{prefix}q{state:<4} | "
            row_cells = []
            for sym in sorted_symbols:
                dest = self.transition(state, sym)
                cell = f"q{dest}" if dest != self.ERROR_STATE else "--"
                row_cells.append(f"{cell:^5}")
            row_str += " | ".join(row_cells)
            print(row_str)

# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN (DESAFÍO 1) ---
if __name__ == "__main__":
    # Modificación del autómata para reconocer:
    # 1. Identificadores: L (L | D)*  -> q1 es estado de aceptación
    # 2. Enteros positivos: D+        -> q2 es estado de aceptación

    states_set = {0, 1, 2}
    alphabet_set = {'L', 'D'}

    # Transiciones del nuevo AFD:
    # q0 --L--> q1 (Inicia identificador)
    # q0 --D--> q2 (Inicia número entero positivo)
    # q1 --L--> q1 (Continúa identificador)
    # q1 --D--> q1 (Continúa identificador)
    # q2 --D--> q2 (Continúa número entero positivo)
    # Nota: Si q2 recibe 'L' o q1 recibe símbolos no definidos, va implícitamente a ERROR_STATE (-1)

    transitions_map = {
        (0, 'L'): 1,
        (0, 'D'): 2,
        (1, 'L'): 1,
        (1, 'D'): 1,
        (2, 'D'): 2
    }

    initial = 0
    accepting = {1, 2} # Tanto q1 como q2 son estados finales de aceptación

    afd_modificado = AutomataFinitoDeterminista(states_set, alphabet_set, transitions_map, initial, accepting)

    # Mostrar la matriz de transición actualizada
    afd_modificado.print_transition_matrix()

    # Ampliación de pruebas (Incluyendo cadenas válidas que comienzan con dígitos, mixtas e inválidas)
    test_inputs = [
        "LLD",   # Válida (Identificador)
        "123",   # Válida que comienza con dígitos (Número entero positivo D+)
        "123L",  # Inválida (Un número seguido de letra no es reconocido por ninguna regla combinada en este diseño simple)
        "L"      # Válida (Identificador de un solo carácter)
    ]

    print("\n=== EVALUACIÓN DE CADENAS DE PRUEBA (DESAFÍO 1) ===")
    for test in test_inputs:
        # Convertir caracteres ('L' para letras, 'D' para dígitos)
        mapped_input = "".join(['L' if c.isalpha() else ('D' if c.isdigit() else c) for c in test])
        print(f"\nEvaluando Entrada Original: '{test}' -> Abstracta: '{mapped_input}'")
        accepted, execution_trace = afd_modificado.evaluate_string(mapped_input)
        for line in execution_trace:
            print(line)


=== MATRIZ DE TRANSICIÓN Dtran[s, c] ===
Estado   |   D   |   L  
------------------------
-> q0    |  q2   |  q1  
 * q1    |  q1   |  q1  
 * q2    |  q2   |  --  

=== EVALUACIÓN DE CADENAS DE PRUEBA (DESAFÍO 1) ===

Evaluando Entrada Original: 'LLD' -> Abstracta: 'LLL'
Estado Inicial: q0
   Paso 1: Símbolo 'L' | Transición q0 -> q1
   Paso 2: Símbolo 'L' | Transición q1 -> q1
   Paso 3: Símbolo 'L' | Transición q1 -> q1
Estado Final: q1 | Resultado: CADENA ACEPTADA

Evaluando Entrada Original: '123' -> Abstracta: 'DDD'
Estado Inicial: q0
   Paso 1: Símbolo 'D' | Transición q0 -> q2
   Paso 2: Símbolo 'D' | Transición q2 -> q2
   Paso 3: Símbolo 'D' | Transición q2 -> q2
Estado Final: q2 | Resultado: CADENA ACEPTADA

Evaluando Entrada Original: '123L' -> Abstracta: 'DDDL'
Estado Inicial: q0
   Paso 1: Símbolo 'D' | Transición q0 -> q2
   Paso 2: Símbolo 'D' | Transición q2 -> q2
   Paso 3: Símbolo 'D' | Transición q2 -> q2
   [Bloqueo] Sin transición desde q2 con 'L'.

Evaluando En

##Desafío 2: Rastreo de Pila y Clausuras Epsilon en AFN (Nivel: Básico-Intermedio)  

Script de referencia: epsilonClosure.py  

**Objetivo:** Analizar el algoritmo basado en pilas (DFS) para calcular la épsilon-clausura y entender la simulación de caminos múltiples no deterministas.  

**Consigna para el estudiante:**  

>Trazado manual vs. computacional: Siga el rastro impreso por el emulador para la cadena `"aabb"`. Identifique visualmente en qué momento exacto el conjunto de estados activos se bifurca gracias a las transiciones $\varepsilon$.  

>Instrumentación del código: Modifique el método `epsilon_closure_state` para que imprima en consola cada vez que un estado entra y sale de la pila (`stack`). Esto le permitirá visualizar el recorrido tipo DFS (Depth-First Search).  

>Prueba de estrés de diseño: Cambie la cadena de prueba a `"aa"` y a `"abbb"`. Analice la traza de ejecución e identifique en qué paso exacto ocurre un `[Bloqueo]` o por qué se rechaza la cadena según el criterio de intersección con el conjunto de aceptación `F`.

**Trazado manual vs. computacional ("aabb")**

Momento	Conjunto activo
Inicial: ε-closure(0)	{0, 1, 2, 4, 7}

Paso 1 a	mover → {3, 8}; ε-closure → {1, 2, 3, 4, 6, 7, 8}

Paso 2 a	igual que el paso 1

Paso 3 b	mover → {5, 9}; ε-closure → {1, 2, 4, 5, 6, 7, 9}

Paso 4 b	mover → {5, 10}; ε-closure → {1, 2, 4, 5, 6, 7, 10}

Primera bifurcación: ocurre antes de leer ningún carácter, en la ε-closure inicial. El estado 0 tiene dos transiciones ε (a 1 y a 7), y el estado 1 tiene otras dos (a 2 y a 4). Así el único estado 0 se convierte en cinco estados activos: {0, 1, 2, 4, 7}.

Bifurcación al consumir símbolos: al leer la primera a, mover produce {3, 8}. Son dos caminos simultáneos: el 3 sigue en el bucle (a|b)* y el 8 avanza hacia abb. Luego el estado 3 pasa por 6 y reabre el bucle con ε (6→1, 6→7), lo que reactiva 1, 2, 4 y 7.

Aceptación: el estado 10 solo aparece en el paso 4, y esa es la prueba de que el camino 7→8→9→10 sobrevivió hasta el final.

**Modificacion epsilon closure state**

    def epsilon_closure_state(self, state: int) -> Set[int]:
        """e-cerradura de un estado con DFS (pila), con traza de PUSH/POP."""
        closure = {state}
        stack = [state]
        if self.verbose:
            print(f"      [PUSH] {state}  (estado semilla) | pila = {stack}")

        while stack:
            current = stack.pop()  # LIFO: se explora en profundidad
            if self.verbose:
                print(f"      [POP ] {current} | pila = {stack}")
            next_states = self.delta.get((current, self.EPSILON), set())
            # sorted() solo para que la traza sea determinista y reproducible
            for nxt in sorted(next_states):
                if nxt not in closure:  # evita ciclos (ej. 6 -> 1 -> ... -> 6)
                    closure.add(nxt)
                    stack.append(nxt)
                    if self.verbose:
                        print(f"      [PUSH] {nxt}  (ε desde {current}) | pila = {stack}")
                elif self.verbose:
                    print(f"      [SKIP] {nxt}  (ya visitado, ε desde {current})")
        return closure



In [ ]:
from typing import Set, Dict, Tuple, List, Any


class NFA_Epsilon:
    """Clase orientada a objetos para modelar y simular un AFN con transiciones epsilon."""
    EPSILON = 'ε'  # Cadena vacía constante

    def __init__(self, states: Set[int], alphabet: Set[str],
                 transitions: Dict[Tuple[int, str], Set[int]],
                 initial_state: int, accepting_states: Set[int]):
        self.S = states
        self.Sigma = alphabet
        self.delta = transitions
        self.s0 = initial_state
        self.F = accepting_states

    def epsilon_closure_state(self, state: int, verbose: bool = False) -> Set[int]:
        """Calcula la ε-cerradura de un estado con DFS iterativo (pila explícita).
        Si verbose=True, imprime cada entrada/salida de la pila."""
        closure = {state}
        stack = [state]
        if verbose:
            print(f"    [PUSH] {state} (inicial)      pila={stack}")

        while stack:
            current = stack.pop()  # LIFO -> recorrido en profundidad
            if verbose:
                print(f"    [POP ] {current}              pila={stack}")

            next_states = self.delta.get((current, self.EPSILON), set())
            if verbose and next_states:
                print(f"           ε-transiciones de {current}: {sorted(next_states)}")

            # sorted() hace determinista el orden (los sets no garantizan orden)
            for nxt in sorted(next_states):
                if nxt not in closure:  # evita ciclos (p. ej. 6 -> 1 -> ... -> 6)
                    closure.add(nxt)
                    stack.append(nxt)
                    if verbose:
                        print(f"    [PUSH] {nxt}              pila={stack}")
        return closure

    def epsilon_closure_set(self, states_set: Set[int]) -> Set[int]:
        """Calcula la epsilon-cerradura para un conjunto de estados: e-closure(T) = U e-closure(s)."""
        total_closure = set()
        for state in states_set:
            total_closure.update(self.epsilon_closure_state(state))
        return total_closure

    def move(self, states_set: Set[int], symbol: str) -> Set[int]:
        """Calcula el conjunto mover(T, a): estados alcanzables con el símbolo 'a'."""
        reachable = set()
        for state in states_set:
            nxt = self.delta.get((state, symbol), set())
            reachable.update(nxt)
        return reachable

    def simulate(self, input_string: str) -> Tuple[bool, List[str]]:
        """Simula la ejecución del AFN (Algoritmo 3.22 del Dragon Book) sobre una cadena."""
        trace = []
        # Paso 1: Estado inicial activo S = e-closure(s0)
        current_active = self.epsilon_closure_state(self.s0)
        trace.append(f"Estado Activo Inicial e-closure({self.s0}): {sorted(list(current_active))}")

        # Bucle principal sobre cada carácter de la entrada
        for i, char in enumerate(input_string):
            if char not in self.Sigma:
                trace.append(f"  [Error] Carácter '{char}' fuera del alfabeto.")
                return False, trace

            # Paso 2a: T = mover(S, c)
            move_set = self.move(current_active, char)
            # Paso 2b: S' = e-closure(T)
            next_active = self.epsilon_closure_set(move_set)

            trace.append(
                f"  Paso {i + 1} ('{char}'): mover -> {sorted(list(move_set))} | e-closure -> {sorted(list(next_active))}")
            current_active = next_active

            if not current_active:
                trace.append("  [Bloqueo] El conjunto de estados activos quedó vacío.")
                return False, trace

        # Criterio de Aceptación: S ∩ F != Ø
        is_accepted = bool(current_active.intersection(self.F))
        status = "CADENA ACEPTADA" if is_accepted else "CADENA RECHAZADA"
        trace.append(
            f"Estados Activos Finales: {sorted(list(current_active))} | Intersección F: {sorted(list(current_active.intersection(self.F)))}")
        trace.append(f"Resultado: {status}")

        return is_accepted, trace


# --- DEMOSTRACIÓN PRÁCTICA Y EVALUACIÓN ---
if __name__ == "__main__":
    # Construcción del AFN-e para el lenguaje (a|b)*abb
    states_all = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
    alphabet_in = {'a', 'b'}

    # Transiciones del AFN-e (basado en la figura 3.34 de Aho et al.)
    transitions_nfa = {
        (0, NFA_Epsilon.EPSILON): {1, 7},
        (1, NFA_Epsilon.EPSILON): {2, 4},
        (2, 'a'): {3},
        (3, NFA_Epsilon.EPSILON): {6},
        (4, 'b'): {5},
        (5, NFA_Epsilon.EPSILON): {6},
        (6, NFA_Epsilon.EPSILON): {1, 7},
        (7, 'a'): {8},
        (8, 'b'): {9},
        (9, 'b'): {10}
    }

    start_st = 0
    accept_st = {10}

    afn = NFA_Epsilon(states_all, alphabet_in, transitions_nfa, start_st, accept_st)

    print("=== EMULADOR DE AFN-e (Simulación Directa) ===")
    test_string = "aabb"
    print(f"\nEvaluando la cadena: '{test_string}'")

    accepted, trace_log = afn.simulate(test_string)
    for line in trace_log:
        print(line)

=== EMULADOR DE AFN-e (Simulación Directa) ===

Evaluando la cadena: 'aabb'
Estado Activo Inicial e-closure(0): [0, 1, 2, 4, 7]
  Paso 1 ('a'): mover -> [3, 8] | e-closure -> [1, 2, 3, 4, 6, 7, 8]
  Paso 2 ('a'): mover -> [3, 8] | e-closure -> [1, 2, 3, 4, 6, 7, 8]
  Paso 3 ('b'): mover -> [5, 9] | e-closure -> [1, 2, 4, 5, 6, 7, 9]
  Paso 4 ('b'): mover -> [5, 10] | e-closure -> [1, 2, 4, 5, 6, 7, 10]
Estados Activos Finales: [1, 2, 4, 5, 6, 7, 10] | Intersección F: [10]
Resultado: CADENA ACEPTADA


### Desafío 3: Construcción de Subconjuntos y Equivalencia de Autómatas (Nivel: Intermedio)  

Script de referencia: afnafd.py y AFDeterminista.py  

**Objetivo:** Conectar la teoría del paso de AFN a AFD mediante el algoritmo de construcción de subconjuntos, analizando la explosión o optimización de estados.  

**Consigna para el estudiante:**  

>Análisis de salida: Ejecute `afnafd.py` y examine la correspondencia entre los subconjuntos numéricos generados (p. ej., `{0, 1, 2, 4, 7}`) y las etiquetas alfabéticas asignadas (`A`, `B`, `C`, etc.).  

>Validación cruzada: Tome la matriz de transición resultante (`d_tran`) generada por el script de subconjuntos para el lenguaje $(a\vert{}b)^*abb$ y conviértala manualmente a una tabla tabular similar a la que imprime `AFDeterminista.py`.  

>Pregunta de reflexión técnica: En su reporte de práctica, responda: ¿Por qué el algoritmo utiliza una cola de índices no marcados (`unmarked_indices`) y qué pasaría con la ejecución si omitimos la verificación `if u_set in d_states`?

### Desafío 4: Extensión de la Construcción de Thompson (Nivel: Intermedio)  

Script de referencia: erafnEpsilon.py  

**Objetivo:** Entender la programación orientada a objetos aplicada a estructuras de grafos y la aplicación práctica de reglas inductivas en compiladores.  

**Consigna para el estudiante:**  

>Estudio de arquitectura POO: Analice cómo la clase `State` gestiona las referencias a objetos vecinos mediante listas de adyacencia y cómo la clase `ThompsonNFA` encapsula los puntos de entrada (`start`) y salida (`accept`).  

>Desafío de implementación (Operador Opcional `?`): Añada un nuevo método estático a la clase `ThompsonNFA` llamado `build_optional(nfa: 'ThompsonNFA') -> 'ThompsonNFA'` que implemente la regla inductiva para la cerradura opcional ($r?$, es decir, cero o una repetición). Pista: requiere conectar el nuevo estado inicial con el estado inicial del sub-autómata y directamente con el nuevo estado final mediante épsilon, además de conectar el final anterior al nuevo final.  

>Prueba de integración: Utilice su nuevo método para construir el AFN correspondiente a la expresión regular opcional `a?` y ejecute la impresión de la lista de adyacencia para verificar sus conexiones.

### Formato de Entrega / Portafolio de Evidencias
1. Trabajen en equipos de dos personas, entregando como reporte en el portafolio de evidencias:

*   El código fuente modificado con comentarios explicativos.

*   Capturas de pantalla o salidas de consola de las pruebas realizadas.

*   Respuesta conceptual a las preguntas de análisis lógico integradas en cada desafío.

2. Guarda este cuaderno con todas las salidas ejecutadas (`Archivo` -> `Guardar una copia en GitHub` / `PEREZ JUAN PSSB I Colab U3.ipynb`).

3. Registra en **Moodle** el enlace del cuaderno ejecutable en Colab y el de tu repositorio conteniendo la especificación formal del proyecto.